# Proyecto: Grupo Importaciones San José S.R.L.
## Avance 3 (APF3) - Predicción de Demanda Semanal por Sede
**Curso:** Innovación y Transformación Digital  
**Docente:** Econ. Richard Camán Sánchez  
**Integrantes:**
- Frank Erick Campomanes Vergara
- Huayaney Lázaro José Luis
- Ronal Dany Hurtado Lorenzo
- Kevin Rosales Álvarez
- Leonardo Adrián García Muñoz

### 1. Cargar librerías y configurar el entorno
Instalamos y cargamos las librerías necesarias para procesar los datos, graficar y entrenar los modelos de regresión.


In [ ]:
!pip install --quiet scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Fijamos la semilla para que los resultados salgan siempre iguales al ejecutar
semilla = 42
np.random.seed(semilla)


### 2. Cargar los datos de ventas y revisar el origen
Leemos el archivo CSV extraído de nuestra base de datos en Aiven. Aquí revisamos cuántas filas tenemos, las fechas y confirmamos que no hay datos personales de clientes (cumpliendo con la Ley 29733). También vemos la columna origen que separa ventas reales de las simuladas.


In [ ]:
archivo = 'dataset_demanda_san_jose.csv'

if not os.path.exists(archivo):
    print('Descargando dataset de ventas...')
    !curl -s -O https://raw.githubusercontent.com/FrankCampomanes/Importaciones_San_Jose/main/dataset_demanda_san_jose.csv

df = pd.read_csv(archivo)
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])

print(f'Total de registros cargados: {len(df)}')
fecha_inicio = df['fecha_hora'].min().strftime('%d/%m/%Y')
fecha_fin = df['fecha_hora'].max().strftime('%d/%m/%Y')
print(f'Periodo registrado: del {fecha_inicio} al {fecha_fin}')

print('\nConteo por tipo de dato en la columna origen:')
print(df['origen'].value_counts())

print('\nRevision de valores nulos por columna:')
print(df.isnull().sum())

df.head()


### 3. ¿Qué categorías se venden más en las tiendas?
Hacemos un gráfico para ver la cantidad total de unidades vendidas según el rubro de producto.


In [ ]:
ventas_categoria = df.groupby('categoria')['cantidad'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 4.5))
barras = plt.barh(ventas_categoria['categoria'], ventas_categoria['cantidad'], color='#2b5c8f')
plt.xlabel('Unidades vendidas en total')
plt.title('Ventas totales por categoría de producto')
plt.gca().invert_yaxis()

for b in barras:
    ancho = b.get_width()
    plt.text(ancho + 15, b.get_y() + b.get_height()/2, f'{int(ancho)} unds', va='center', fontsize=9)

plt.tight_layout()
plt.show()

top_cat = ventas_categoria.iloc[0]['categoria']
print(f'Categoría con mayor volumen de ventas: {top_cat}')


### 4. Comparación de ventas: lunes a viernes vs fin de semana
Revisamos cómo se distribuyen las compras a lo largo de los días de la semana para comprobar si los fines de semana hay picos de demanda.


In [ ]:
dias_ingles = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dias_espanol = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
traductor_dias = dict(zip(dias_ingles, dias_espanol))

df['dia_nombre'] = df['dia_semana'].map(traductor_dias)

resumen_dias = df.groupby('dia_nombre').agg(
    total_unidades=('cantidad', 'sum'),
    es_fin_de_semana=('es_fin_de_semana', 'first')
).reindex(dias_espanol).reset_index()

colores = ['#4682b4' if fin == 0 else '#e2583e' for fin in resumen_dias['es_fin_de_semana']]

plt.figure(figsize=(9, 4.5))
barras = plt.bar(resumen_dias['dia_nombre'], resumen_dias['total_unidades'], color=colores)
plt.ylabel('Unidades vendidas')
plt.title('Ventas totales por día de la semana', fontsize=12, weight='bold')

for b in barras:
    alto = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, alto + 10, f'{int(alto)}', ha='center', fontsize=9)

# Leyenda clara con colores abajo
import matplotlib.patches as mpatches
parche_azul = mpatches.Patch(color='#4682b4', label='Lunes a Viernes (Días laborables)')
parche_naranja = mpatches.Patch(color='#e2583e', label='Sábado y Domingo (Fin de semana)')
plt.legend(handles=[parche_azul, parche_naranja], loc='upper left')

plt.tight_layout()
plt.show()


### 5. Agrupar las ventas por semana (crear la variable objetivo)
La supervisora toma decisiones una vez por semana. Por eso agrupamos las ventas por semana del año, sede y producto, sumando las unidades para obtener la demanda semanal.


In [ ]:
# Agrupamos por semana, tienda y producto
df_semanal = df.groupby(
    ['semana_iso', 'id_sede', 'nombre_sede', 'id_producto', 'nombre_producto', 'categoria']
).agg(
    demanda_semanal=('cantidad', 'sum'),
    precio_promedio=('precio_unitario', 'mean'),
    costo_promedio=('costo_unitario', 'mean'),
    proporcion_fin_semana=('es_fin_de_semana', 'mean')
).reset_index()

# Margen de ganancia en porcentaje
df_semanal['margen_porcentual'] = (df_semanal['precio_promedio'] - df_semanal['costo_promedio']) / df_semanal['precio_promedio']
df_semanal['margen_porcentual'] = df_semanal['margen_porcentual'].fillna(0.35)

# Ordenamos por tienda, producto y semana
df_semanal = df_semanal.sort_values(by=['id_sede', 'id_producto', 'semana_iso']).reset_index(drop=True)

# Guardamos la venta de la semana anterior para que el modelo la use como referencia
df_semanal['demanda_semana_anterior'] = df_semanal.groupby(['id_sede', 'id_producto'])['demanda_semanal'].shift(1)
df_semanal['demanda_semana_anterior'] = df_semanal['demanda_semana_anterior'].fillna(df_semanal['demanda_semanal'].median())

print(f"Total de registros semanales por producto y sede: {len(df_semanal)}")
df_semanal.head()


### 6. Elegir las columnas para el modelo y preparar el preprocesamiento
Seleccionamos qué datos numéricos y de texto usará el modelo para hacer sus cálculos.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

columnas_texto = ['nombre_sede', 'categoria']
columnas_numeros = ['precio_promedio', 'costo_promedio', 'margen_porcentual', 'proporcion_fin_semana', 'demanda_semana_anterior']

X = df_semanal[columnas_texto + columnas_numeros]
y = df_semanal['demanda_semanal']

# Pasamos las categorías a columnas y estandarizamos los números
preprocesador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), columnas_texto),
        ('num', StandardScaler(), columnas_numeros)
    ]
)


### 7. Separar los datos por fechas (70% entrenamiento, 15% validación, 15% prueba)
Como esto es una serie de tiempo, cortamos por semanas y no al azar, para que el modelo no aprenda con datos del futuro.


In [ ]:
semanas = sorted(df_semanal['semana_iso'].unique())
total_filas = len(df_semanal)

# Usamos las primeras semanas para entrenar, luego validación y dejamos las últimas para prueba
corte_1 = int(total_filas * 0.70)
corte_2 = int(total_filas * 0.85)

X_train = X.iloc[:corte_1]
y_train = y.iloc[:corte_1]

X_val = X.iloc[corte_1:corte_2]
y_val = y.iloc[corte_1:corte_2]

X_test = X.iloc[corte_2:]
y_test = y.iloc[corte_2:]

print(f"Datos para entrenar: {len(X_train)} filas")
print(f"Datos para validar: {len(X_val)} filas")
print(f"Datos de prueba final: {len(X_test)} filas")


### 8. Línea base: el cálculo tradicional del supervisor
Medimos qué tan bien o mal le va a la regla simple de enviar lo mismo que se vendió la semana pasada. Esta es la referencia que nuestros modelos de machine learning deben mejorar.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar(y_real, y_pred, nombre):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    wape = (np.sum(np.abs(y_real - y_pred)) / np.sum(y_real)) * 100
    r2 = r2_score(y_real, y_pred)
    return {'Modelo': nombre, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'WAPE (%)': round(wape, 2), 'R2': round(r2, 2)}

# El pronóstico tradicional usa la venta de la semana anterior
pred_base_test = X_test['demanda_semana_anterior']
resultado_base = evaluar(y_test, pred_base_test, "Línea Base (Semana anterior)")

print("Resultados de la regla del supervisor:")
print(f"MAE: {resultado_base['MAE']} unidades")
print(f"WAPE: {resultado_base['WAPE (%)']}%")


### 9. Modelo 1: Regresión Lineal Ridge
Probamos una regresión lineal con regularización Ridge para mantener los coeficientes controlados y evitar que se disparen con números altos.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

modelo_ridge = Pipeline([
    ('pre', preprocesador),
    ('reg', Ridge(alpha=1.0, random_state=semilla))
])

modelo_ridge.fit(X_train, y_train)
pred_ridge_val = modelo_ridge.predict(X_val)

res_ridge = evaluar(y_val, pred_ridge_val, "Ridge")
print(f"Ridge en validación -> MAE: {res_ridge['MAE']} unds | WAPE: {res_ridge['WAPE (%)']}%")


### 10. Modelo 2: KNN sin normalizar los datos
Probamos el algoritmo de vecinos más cercanos sin escalar los precios ni las cantidades. Nos sirve para ver qué pasa cuando dejamos los números en su escala original.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

preprocesador_simple = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), columnas_texto),
        ('num', 'passthrough', columnas_numeros)
    ]
)

modelo_knn_sin_escalar = Pipeline([
    ('pre', preprocesador_simple),
    ('reg', KNeighborsRegressor(n_neighbors=5))
])

modelo_knn_sin_escalar.fit(X_train, y_train)
pred_knn_val = modelo_knn_sin_escalar.predict(X_val)

res_knn_raw = evaluar(y_val, pred_knn_val, "KNN sin escalar")
print(f"KNN sin escalar en validación -> MAE: {res_knn_raw['MAE']} unds | WAPE: {res_knn_raw['WAPE (%)']}%")


### 11. Modelo 3: KNN con StandardScaler
Ahora probamos KNN pero normalizando todas las columnas numéricas con StandardScaler. Así comprobamos en la práctica cómo mejora el algoritmo cuando todos los datos están en la misma escala.


In [ ]:
modelo_knn_escalado = Pipeline([
    ('pre', preprocesador),
    ('reg', KNeighborsRegressor(n_neighbors=5, weights='distance'))
])

modelo_knn_escalado.fit(X_train, y_train)
pred_knn_esc_val = modelo_knn_escalado.predict(X_val)

res_knn_esc = evaluar(y_val, pred_knn_esc_val, "KNN con StandardScaler")
print(f"KNN escalado en validación -> MAE: {res_knn_esc['MAE']} unds | WAPE: {res_knn_esc['WAPE (%)']}%")
print(f"Diferencia de error gracias a la normalización: {res_knn_raw['MAE'] - res_knn_esc['MAE']:.2f} unidades menos.")


### 12. Modelo 4: Árbol de Decisión con poda
Entrenamos un árbol de decisión limitando su profundidad a 4 niveles. Esto evita que el árbol memorice casos raros y permite entender las reglas con claridad.


In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

modelo_arbol = Pipeline([
    ('pre', preprocesador),
    ('reg', DecisionTreeRegressor(max_depth=4, min_samples_leaf=4, random_state=semilla))
])

modelo_arbol.fit(X_train, y_train)
pred_arbol_val = modelo_arbol.predict(X_val)

res_arbol = evaluar(y_val, pred_arbol_val, 'Árbol de Decisión')
print(f'Árbol podado en validación: MAE = {res_arbol["MAE"]} unds | WAPE = {res_arbol["WAPE (%)"]}%')

# Diagrama visual del árbol de decisión para el informe
plt.figure(figsize=(14, 6))
nombres_features = preprocesador.get_feature_names_out()
plot_tree(modelo_arbol.named_steps['reg'], max_depth=2, feature_names=nombres_features, filled=True, rounded=True, fontsize=8)
plt.title('Diagrama de decisiones del árbol (primeros niveles de preguntas lógicas)', fontsize=11, weight='bold')
plt.tight_layout()
plt.show()


### 13. Modelo 5: Gradient Boosting
Entrenamos un ensamble de Gradient Boosting, que va creando pequeños árboles donde cada uno se enfoca en corregir los errores que dejó el árbol anterior.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

modelo_gb = Pipeline([
    ('pre', preprocesador),
    ('reg', GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=semilla
    ))
])

modelo_gb.fit(X_train, y_train)
pred_gb_val = modelo_gb.predict(X_val)

res_gb = evaluar(y_val, pred_gb_val, "Gradient Boosting")
print(f"Gradient Boosting en validación -> MAE: {res_gb['MAE']} unds | WAPE: {res_gb['WAPE (%)']}%")


### 14. Comparación final de todos los modelos en los datos de prueba
Ponemos a competir a todos los modelos frente a los datos de prueba que guardamos al inicio y que ninguno vio durante el entrenamiento.


In [ ]:
modelos_a_probar = [
    ('Línea Base (Supervisor)', pred_base_test),
    ('KNN sin normalizar', modelo_knn_sin_escalar.predict(X_test)),
    ('KNN con StandardScaler', modelo_knn_escalado.predict(X_test)),
    ('Regresión Ridge', modelo_ridge.predict(X_test)),
    ('Árbol de Decisión', modelo_arbol.predict(X_test)),
    ('Gradient Boosting', modelo_gb.predict(X_test))
]

resumen_torneo = []
for nombre, predicciones in modelos_a_probar:
    metricas = evaluar(y_test, predicciones, nombre)
    if nombre != 'Línea Base (Supervisor)':
        mejora = ((resultado_base['MAE'] - metricas['MAE']) / resultado_base['MAE']) * 100
        metricas['Mejora (%)'] = round(mejora, 1)
    else:
        metricas['Mejora (%)'] = 0.0
    resumen_torneo.append(metricas)

tabla_resultados = pd.DataFrame(resumen_torneo)
tabla_resultados = tabla_resultados.rename(columns={
    'MAE': 'Error MAE (unidades)',
    'RMSE': 'Error RMSE',
    'WAPE (%)': 'Error porcentual WAPE',
    'R2': 'Precisión R²'
})

print('Tabla comparativa final en datos de prueba:')
display(tabla_resultados)

# Graficamos el MAE de cada modelo
plt.figure(figsize=(10, 4.5))
colores = ['#888888', '#d9534f', '#f0ad4e', '#5bc0de', '#0275d8', '#5cb85c']
barras = plt.bar(tabla_resultados['Modelo'], tabla_resultados['Error MAE (unidades)'], color=colores, edgecolor='black')
plt.ylabel('Error promedio en unidades físicas')
plt.title('Comparación de error promedio por modelo (menor barra = más exacto)', fontsize=11, weight='bold')
plt.xticks(rotation=20, ha='right')

for b in barras:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, h + 0.05, f'{h:.2f} unds', ha='center', fontsize=9, weight='bold')

plt.tight_layout()
plt.show()


### 15. Revisión de errores y casos atípicos
Revisamos en qué productos o situaciones el modelo ganador cometió los errores más grandes y planteamos hipótesis basadas en la realidad de la tienda.


In [ ]:
pred_final = modelo_gb.predict(X_test)
errores = y_test - pred_final

analisis_fallos = X_test.copy()
analisis_fallos['Venta_Real'] = y_test.values
analisis_fallos['Venta_Predicha'] = np.round(pred_final, 1)
analisis_fallos['Error_Unidades'] = np.round(np.abs(errores).values, 1)

mayores_fallos = analisis_fallos.sort_values(by='Error_Unidades', ascending=False).head(3)

print('Los 3 mayores errores encontrados en la prueba:')
for i, (_, fila) in enumerate(mayores_fallos.iterrows(), 1):
    cat = fila['categoria']
    sede = fila['nombre_sede']
    vr = fila['Venta_Real']
    vp = fila['Venta_Predicha']
    eu = fila['Error_Unidades']
    print(f'Caso {i}: {cat} en {sede} -> Venta real: {vr} unds | Predicción: {vp} unds | Error: {eu} unds')

print('\nPosibles explicaciones de estos casos en tienda:')
print('1. Compras por docena: Clientes que compran un lote cerrado en un solo momento.')
print('2. Imprevistos en tienda: Cierres temporales en sedes por arreglos de pistas o cortes en Lima Norte.')
print('3. Picos por fin de mes: Coincidencia de pagos de quincena con festividades locales.')


### 16. ¿Qué factores pesan más en la predicción?
Miramos qué variables le sirvieron más a Gradient Boosting para acertar en la demanda semanal.


In [ ]:
nombres_cat = modelo_gb.named_steps['pre'].named_transformers_['cat'].get_feature_names_out(columnas_texto)
todas_columnas = list(nombres_cat) + columnas_numeros
importancias = modelo_gb.named_steps['reg'].feature_importances_

df_pesos = pd.DataFrame({
    'Factor': todas_columnas,
    'Peso': importancias
}).sort_values(by='Peso', ascending=True).tail(7)

plt.figure(figsize=(9, 4))
plt.barh(df_pesos['Factor'], df_pesos['Peso'], color='#2a9d8f')
plt.xlabel('Importancia relativa en la decisión')
plt.title('Variables que más influyen en el pronóstico de demanda')
plt.tight_layout()
plt.show()

print("La venta de la semana anterior y la sede elegida son los dos factores de mayor peso para el pronóstico.")


### 17. Guardar el modelo ganador para usarlo luego
Exportamos el pipeline entrenado en un archivo `.joblib` para poder cargarlo en cualquier momento sin tener que volver a entrenar.


In [ ]:
archivo_modelo = 'modelo_demanda_san_jose.joblib'
joblib.dump(modelo_gb, archivo_modelo)

peso_kb = os.path.getsize(archivo_modelo) / 1024
print(f"Modelo guardado como '{archivo_modelo}' con un peso de {peso_kb:.1f} KB.")


### 18. Simulación interactiva: calcular el pedido para una sede
Esta función simula lo que vería la supervisora central al consultar cuánta mercadería debe mandar a una tienda específica.


In [ ]:
archivo_modelo = 'modelo_demanda_san_jose.joblib'

# Aseguramos el archivo antes de cargarlo
if not os.path.exists(archivo_modelo):
    joblib.dump(modelo_gb, archivo_modelo)

modelo_guardado = joblib.load(archivo_modelo)

def consultar_pedido(sede, categoria, precio, costo, venta_semana_anterior, es_fin_semana=1):
    margen = (precio - costo) / precio
    
    datos_consulta = pd.DataFrame([{
        'nombre_sede': sede,
        'categoria': categoria,
        'precio_promedio': precio,
        'costo_promedio': costo,
        'margen_porcentual': margen,
        'proporcion_fin_semana': 1.0 if es_fin_semana else 0.0,
        'demanda_semana_anterior': venta_semana_anterior
    }])
    
    demanda = modelo_guardado.predict(datos_consulta)[0]
    unidades = int(np.ceil(demanda))
    stock_colchon = max(2, int(np.ceil(unidades * 0.20)))
    total_despacho = unidades + stock_colchon
    
    print('-' * 55)
    print('RECOMENDACIÓN DE REPOSICIÓN - IMPORTACIONES SAN JOSÉ')
    print('-' * 55)
    print(f'Sede:                   {sede}')
    print(f'Categoría:              {categoria}')
    print(f'Precio:                 S/. {precio:.2f} (Costo: S/. {costo:.2f})')
    print(f'Venta semana anterior:  {venta_semana_anterior} unidades')
    print('-' * 55)
    print(f'Demanda esperada:       {demanda:.1f} unidades')
    print(f'Margen de seguridad:    {stock_colchon} unidades')
    print(f'TOTAL A DESPACHAR:      {total_despacho} unidades')
    print('-' * 55)

consultar_pedido('Sede Los Olivos', 'Belleza y Cuidado Personal', 3.50, 1.80, 15)


### 19. Aspectos normativos y supervisión humana
Declaramos cómo cumple el proyecto las normativas peruanas de protección de datos (Ley 29733) y de inteligencia artificial (Ley 31814).


In [ ]:
normativa = pd.DataFrame([
    {
        "Norma": "Ley N.° 29733 (Datos Personales)",
        "Aplicación en el proyecto": "No se usan nombres, DNIs, teléfonos ni correos de clientes en el dataset.",
        "Resultado": "Se trabaja solo con cantidades y productos a nivel sede."
    },
    {
        "Norma": "Ley N.° 31814 (Inteligencia Artificial)",
        "Aplicación en el proyecto": "Supervisión humana: el modelo genera una sugerencia pero no despacha sola.",
        "Resultado": "La supervisora en Puente Piedra revisa y aprueba el traslado final."
    },
    {
        "Norma": "Ley N.° 28015 (Ley MYPE)",
        "Aplicación en el proyecto": "El cálculo es rápido y no añade trámites engorrosos a la tienda.",
        "Resultado": "Respuesta en milisegundos para apoyar la decisión de reposición."
    }
])

display(normativa)
